In [ ]:
import pandas as pd
import numpy as np

import random
random.seed(42)

from datetime import datetime, timedelta
from google.colab import files

In [ ]:
# Number of locations
NUM_LOCATIONS = 60

In [ ]:
# Hyderabad coordinate boundaries
LAT_MIN = 17.20
LAT_MAX = 17.60

LON_MIN = 78.20
LON_MAX = 78.70

In [ ]:
zones = ["North", "South", "East", "West", "Central"]


In [ ]:
# locations = []

# for i in range(1, NUM_LOCATIONS + 1):

#     location = {
#         "location_id": f"L{i}",
#         "store_name": f"Store_{i}",
#         "latitude": round(random.uniform(LAT_MIN, LAT_MAX), 6),
#         "longitude": round(random.uniform(LON_MIN, LON_MAX), 6),
#         "zone": random.choice(zones)
#     }

#     locations.append(location)

In [ ]:
def generate_locations(num_locations, lat_min, lat_max, lon_min, lon_max,
                       zones_list):
    # random.seed(42)
    locations_list = []
    for i in range(1, num_locations + 1):
        location = {
            "location_id": f"L{i}",
            "store_name": f"Store_{i}",
            "latitude": round(random.uniform(lat_min, lat_max), 6),
            "longitude": round(random.uniform(lon_min, lon_max), 6),
            "zone": random.choice(zones_list),
        }
        locations_list.append(location)
    return locations_list

locations = generate_locations(
    NUM_LOCATIONS, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX, zones
)

In [ ]:
type(locations)

list

In [ ]:
for i in range(len(locations)):
  print(locations[i])

{'location_id': 'L1', 'store_name': 'Store_1', 'latitude': 17.455771, 'longitude': 78.212505, 'zone': 'East'}
{'location_id': 'L2', 'store_name': 'Store_2', 'latitude': 17.297957, 'longitude': 78.269769, 'zone': 'North'}
{'location_id': 'L3', 'store_name': 'Store_3', 'latitude': 17.47068, 'longitude': 78.64609, 'zone': 'North'}
{'location_id': 'L4', 'store_name': 'Store_4', 'latitude': 17.436197, 'longitude': 78.215891, 'zone': 'North'}
{'location_id': 'L5', 'store_name': 'Store_5', 'latitude': 17.287455, 'longitude': 78.452678, 'zone': 'North'}
{'location_id': 'L6', 'store_name': 'Store_6', 'latitude': 17.424498, 'longitude': 78.55801, 'zone': 'Central'}
{'location_id': 'L7', 'store_name': 'Store_7', 'latitude': 17.367808, 'longitude': 78.424605, 'zone': 'East'}
{'location_id': 'L8', 'store_name': 'Store_8', 'latitude': 17.523772, 'longitude': 78.203249, 'zone': 'South'}
{'location_id': 'L9', 'store_name': 'Store_9', 'latitude': 17.479256, 'longitude': 78.370125, 'zone': 'South'}
{'lo

In [ ]:
# Convert to dataframe
locations_df = pd.DataFrame(locations)

# Save CSV
locations_df.to_csv("locations.csv", index=False)

print("Locations dataset generated successfully!")

Locations dataset generated successfully!


In [ ]:
files.download('locations.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Parameters
NUM_DRIVERS = 12
NUM_DAYS = 40

In [ ]:
drivers = [f"D{i}" for i in range(1, NUM_DRIVERS + 1)]

In [ ]:
# Date range
start_date = datetime(2026, 4, 1)

In [ ]:
AVG_SPEED_KMPH = 40

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(selected_locations, i):

    # Last stop has no next location
    if i >= len(selected_locations) - 1:
        return 0

    current_location = selected_locations.iloc[i]

    next_location = selected_locations.iloc[i + 1]

    # Earth radius in KM
    R = 6371

    # Convert coordinates to radians
    lat1, lon1, lat2, lon2 = map(
        radians,
        [
            current_location["latitude"],
            current_location["longitude"],
            next_location["latitude"],
            next_location["longitude"]
        ]
    )

    # Coordinate differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance_km = R * c

    return round(distance_km, 2)

In [ ]:
def generate_trip_stops(
    selected_locations,
    current_time,
    driver,
    trip_id,
    current_date
):
    random.seed(trip_id)

    records = []

    stop_number = 1

    for i, location in selected_locations.iterrows():

        visit_duration = random.randint(10, 40)

        distance_km=haversine_distance(selected_locations,i)


        travel_time_min = (
            distance_km / AVG_SPEED_KMPH
        ) * 60

        travel_time_min = round(travel_time_min, 1)

        trip_record = {

            "trip_id": trip_id,
            "driver_id": driver,
            "date": current_date.date(),

            "stop_number": stop_number,

            "location_id": location["location_id"],
            "store_name": location["store_name"],

            "latitude": location["latitude"],
            "longitude": location["longitude"],

            "arrival_time": current_time.strftime("%H:%M"),

            "visit_duration_min": visit_duration,

            "distance_to_next_km": distance_km,

            "travel_time_to_next_min": travel_time_min,
        }
        records.append(trip_record)

        current_time += timedelta(
            minutes=(
                visit_duration
                + travel_time_min
                + random.randint(5, 15)
            )
        )

        stop_number += 1

    return records

In [ ]:
trip_records = []

trip_id = 1

for day in range(NUM_DAYS):

    current_date = start_date + timedelta(days=day)
    random.seed(day)

    for driver in drivers:

        num_stops = random.randint(3, 7)

        selected_locations = (
            locations_df.sample(num_stops,random_state=42)
            .reset_index(drop=True)
        )

        current_time = datetime.combine(
            current_date.date(),
            datetime.strptime(
                f"{random.randint(8,10)}:{random.choice([0,15,30,45])}",
                "%H:%M"
            ).time()
        )

        driver_trip_records = generate_trip_stops(
            selected_locations,
            current_time,
            driver,
            trip_id,
            current_date
        )

        trip_records.extend(driver_trip_records)

        trip_id += 1

In [ ]:
len(trip_records)

2385

In [ ]:
for i in range(20):
  print(trip_records[i])

{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 1, 'location_id': 'L1', 'store_name': 'Store_1', 'latitude': 17.455771, 'longitude': 78.212505, 'arrival_time': '09:00', 'visit_duration_min': 14, 'distance_to_next_km': 36.82, 'travel_time_to_next_min': 55.2}
{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 2, 'location_id': 'L6', 'store_name': 'Store_6', 'latitude': 17.424498, 'longitude': 78.55801, 'arrival_time': '10:23', 'visit_duration_min': 37, 'distance_to_next_km': 10.9, 'travel_time_to_next_min': 16.4}
{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 3, 'location_id': 'L37', 'store_name': 'Store_37', 'latitude': 17.472684, 'longitude': 78.468485, 'arrival_time': '11:22', 'visit_duration_min': 18, 'distance_to_next_km': 21.2, 'travel_time_to_next_min': 31.8}
{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 4, 'location_id': 'L46', 'store_nam

In [ ]:
for i in range(len(trip_records)):
  print(trip_records[i])

{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 1, 'location_id': 'L1', 'store_name': 'Store_1', 'latitude': 17.455771, 'longitude': 78.212505, 'arrival_time': '09:00', 'visit_duration_min': 14, 'distance_to_next_km': 36.82, 'travel_time_to_next_min': 55.2}
{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 2, 'location_id': 'L6', 'store_name': 'Store_6', 'latitude': 17.424498, 'longitude': 78.55801, 'arrival_time': '10:23', 'visit_duration_min': 37, 'distance_to_next_km': 10.9, 'travel_time_to_next_min': 16.4}
{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 3, 'location_id': 'L37', 'store_name': 'Store_37', 'latitude': 17.472684, 'longitude': 78.468485, 'arrival_time': '11:22', 'visit_duration_min': 18, 'distance_to_next_km': 21.2, 'travel_time_to_next_min': 31.8}
{'trip_id': 1, 'driver_id': 'D1', 'date': datetime.date(2026, 4, 1), 'stop_number': 4, 'location_id': 'L46', 'store_nam

In [ ]:
# Create dataframe
trips_df = pd.DataFrame(trip_records)

# Save dataset
trips_df.to_csv("historical_trips.csv", index=False)

print("Historical trips dataset generated successfully!")

files.download("historical_trips.csv")

Historical trips dataset generated successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def calculate_distance_to_next(lat1, lon1, lat2, lon2):

    R = 6371  # Earth radius in KM

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [ ]:
trip_records = []

trip_id = 1

for day in range(NUM_DAYS):

    current_date = start_date + timedelta(days=day)

    for driver in drivers:

        # Each driver visits 4–8 stores daily
        num_stops = random.randint(3, 10)

        # Select random stores
        selected_locations = (
            locations_df.sample(num_stops,random_state=42)
            .reset_index(drop=True)
        )

        # Driver starts between 8–10 AM
        current_time = datetime.combine(
            current_date.date(),
            datetime.strptime(
                f"{random.randint(8,10)}:{random.choice([0,15,30,45])}",
                "%H:%M"
            ).time()
        )

        stop_number = 1

        for i, location in selected_locations.iterrows():

            visit_duration = random.randint(10, 40)

            if ((i) < (len(selected_locations) - 1)):

              next_location = selected_locations.iloc[i + 1]

              distance_km = calculate_distance_to_next(
                  location["latitude"],
                  location["longitude"],
                  next_location["latitude"],
                  next_location["longitude"]
              )

            else:
              distance_km = 0

            distance_km= round(distance_km, 2)


            travel_time_min = (
                distance_km / AVG_SPEED_KMPH
            ) * 60
            travel_time_min = round(travel_time_min, 1)

            trip_record = {
                # "trip_id": f"T{trip_counter}",
                "trip_id": trip_id,
                "driver_id": driver,
                "date": current_date.date(),
                "stop_number": stop_number,
                "location_id": location["location_id"],
                "store_name": location["store_name"],
                "latitude": location["latitude"],
                "longitude": location["longitude"],
                "arrival_time": current_time.strftime("%H:%M"),
                "visit_duration_min": visit_duration,
                "distance_to_next_km": distance_km,
                "travel_time_to_next_min": travel_time_min,
            }

            trip_records.append(trip_record)


            # Move time forward
            current_time += timedelta(
                minutes=visit_duration + travel_time_min + random.randint(5, 15)
            )

            stop_number += 1

        trip_id += 1

In [ ]:
len(trip_records)


4664